# Spark cluster from JupyterLab

This kernel talks to the standalone cluster on the Compose network:

* master — `spark://spark-master:7077` (driver here in the notebook, executors on `spark-worker`)
* Kafka — `kafka:19092` (the in-cluster listener)

The Kafka connector is injected by `PYSPARK_SUBMIT_ARGS` on the container, so
`spark.read.format("kafka")` works without any import or jar juggling.


In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("jupyter-experiment")
    .master("spark://spark-master:7077")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| master:", spark.sparkContext.master)


## Is the work actually running on the worker?

In [ ]:
import json, urllib.request

sc = spark.sparkContext
total = sc.parallelize(range(1_000_000), 8).map(lambda x: x % 7).sum()
print("sum of range(1e6) mod 7 =", total, "(expected 2999997)")

s = json.loads(urllib.request.urlopen("http://spark-master:8080/json/").read())
print("workers alive:", s["aliveworkers"], "| cores:", s["cores"], "| memory MB:", s["memory"])
print("my app as the master sees it:",
      [(a["name"], a["cores"], a["state"]) for a in s["activeapps"]])


## Write to Kafka, read it back

In [ ]:
BROKER, TOPIC = "kafka:19092", "notebook-events"

(spark.range(50)
   .select(F.concat(F.lit("user-"), (F.col("id") % 4).cast("string")).alias("key"),
           F.to_json(F.struct(F.col("id").alias("n"),
                              (F.col("id") % 4).alias("user"),
                              F.lit("click").alias("type"))).alias("value"))
   .write.format("kafka")
   .option("kafka.bootstrap.servers", BROKER)
   .option("topic", TOPIC)
   .save())
print("wrote 50 records to", TOPIC)


In [ ]:
df = (spark.read.format("kafka")
      .option("kafka.bootstrap.servers", BROKER)
      .option("subscribe", TOPIC)
      .option("startingOffsets", "earliest")
      .option("endingOffsets", "latest")
      .load())

(df.select(F.from_json(F.col("value").cast("string"), "n long, user long, type string").alias("e"))
   .select("e.*")
   .groupBy("user").count().orderBy("user").toPandas())


## Cleanup

Leave the session running to keep experimenting; `spark.stop()` when done.

In [ ]:
spark.stop()
print("session stopped")
